In [1]:
import pickle
import math
import numpy as np
import seaborn as sns

from rl_firm.abm.parameter import ModelParameters, FirmParameters, HouseholdParameters
from rl_firm.abm.firm import Firm, LearningFirm
from rl_firm.abm.household import Household
from rl_firm.abm.model import BaselineModel

NOTE: Creating burned-in snapshots of ABMs to facilitate training/evaluating

There are 2 snapshot version:
- One learning firm for training + firm level evaluation
- 100 learning firms for macro analysis (learning firm can use baseline heuristics for different adoption scenarios)

## Training + firm level evaluation (1 learning firm)

In [2]:
parameters = ModelParameters(
    month_length = 21,
    initial_employment_rate = 0.0,
    n_firms = 100,
    n_learning_firms = 1,
    n_households = 1000,
    burn_in_months = 1000,
    
    firm = FirmParameters(
        gamma_consecutive_months = 24,
        delta_wage_adjustment_rate = 0.019,
        upper_phi_demand = 1,
        lower_phi_demand = 0.25,
        upper_phi_marginal_cost = 1.15,
        lower_phi_marginal_cost = 1.025,
        theta_goods_price = 0.02,
        theta_probs_set_new_price = 0.75,
        lambda_technology = 3,
        chi_buffer = 0.1,

        initial_open_position = True,
        initial_liquidity = 0,
        initial_goods_price = 40,
        initial_wage_rate = 2000,
        #initial_goods_price = 10,
        #initial_wage_rate = 600,
    ),

    household = HouseholdParameters(
        xi_price_percentage = 0.01,
        beta_firm_count = 5,
        pi_explore_odd = 0.1,
        alpha_budget_exponent = 0.9,
        n_suppliers = 7,
        psi_price = 0.25,
        psi_quant = 0.25,

        initial_reservation_wage = 0,
        initial_liquidity = 5000,
    )
)

In [3]:
models = [BaselineModel(parameters=parameters, seed=seed) for seed in range(40)]

In [4]:
%%time

[model.burn_in() for model in models]

CPU times: user 1h 8min 12s, sys: 19.8 s, total: 1h 8min 31s
Wall time: 1h 9min 12s


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [5]:
[[(f.unique_id, f.profit_ranking, round(f.goods_price, 2), round(f.wage_rate, 2), round(f.last_month_profit, 2), round(f.inventories, 2), f.employee_count, f.customer_count) for f in model.firms[-1:]] for model in models]

[[(100, 97, 43.05, 1412.61, 0.0, 0.0, 0, 2)],
 [(100, 59, 42.69, 2202.11, 2309.86, 15.0, 5, 66)],
 [(100, 9, 46.13, 2120.52, 13924.34, 1561.45, 11, 53)],
 [(100, 15, 44.65, 2322.9, 11211.7, 86.32, 24, 74)],
 [(100, 31, 44.71, 2381.74, 7510.93, 1054.13, 16, 58)],
 [(100, 35, 43.47, 2347.82, 5470.35, 42.0, 14, 85)],
 [(100, 33, 43.76, 1609.28, 6584.63, 91.68, 1, 10)],
 [(100, 98, 43.51, 2174.36, 0.0, 0.0, 0, 11)],
 [(100, 3, 46.82, 2358.86, 20802.03, 485.68, 14, 73)],
 [(100, 97, 40.33, 2070.11, 0.0, 0.0, 0, 1)],
 [(100, 96, 44.01, 1591.66, 0.0, 0.0, 0, 9)],
 [(100, 98, 44.45, 1646.43, 0.0, 0.0, 0, 2)],
 [(100, 6, 45.55, 2431.74, 12809.07, 84.0, 28, 349)],
 [(100, 61, 41.58, 1977.24, 1812.06, 3.0, 1, 3)],
 [(100, 98, 44.57, 1899.49, 0, 0.0, 0, 0)],
 [(100, 14, 43.46, 2282.99, 11251.02, 75.0, 25, 210)],
 [(100, 27, 46.86, 2376.44, 6461.17, 33.0, 11, 34)],
 [(100, 29, 45.69, 2430.59, 6097.54, 39.0, 13, 155)],
 [(100, 27, 42.96, 1998.81, 7551.05, 6.0, 2, 16)],
 [(100, 14, 45.32, 2315.14, 10

In [6]:
for model in models:
    # Store the burned in model as pickle to reuse it
    with open(f"../snapshots/abm_{model._seed}.pkl", "wb") as f:
        pickle.dump(model, f)

## Macro analysis model

In [7]:
parameters = ModelParameters(
    month_length = 21,
    initial_employment_rate = 0.0,
    n_firms = 100,
    n_learning_firms = 100,
    n_households = 1000,
    burn_in_months = 1000,
    
    firm = FirmParameters(
        gamma_consecutive_months = 24,
        delta_wage_adjustment_rate = 0.019,
        upper_phi_demand = 1,
        lower_phi_demand = 0.25,
        upper_phi_marginal_cost = 1.15,
        lower_phi_marginal_cost = 1.025,
        theta_goods_price = 0.02,
        theta_probs_set_new_price = 0.75,
        lambda_technology = 3,
        chi_buffer = 0.1,

        initial_open_position = True,
        initial_liquidity = 0,
        initial_goods_price = 40,
        initial_wage_rate = 2000,
    ),

    household = HouseholdParameters(
        xi_price_percentage = 0.01,
        beta_firm_count = 5,
        pi_explore_odd = 0.1,
        alpha_budget_exponent = 0.9,
        n_suppliers = 7,
        psi_price = 0.25,
        psi_quant = 0.25,

        initial_reservation_wage = 0,
        initial_liquidity = 5000,
    )
)

In [8]:
macro_models = [BaselineModel(parameters=parameters, seed=seed) for seed in range(20, 40)]

In [9]:
len(macro_models[0].learning_firms)

100

In [10]:
%%time

[macro_model.burn_in() for macro_model in macro_models]

CPU times: user 33min 49s, sys: 7.14 s, total: 33min 56s
Wall time: 34min 10s


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [11]:
for macro_model in macro_models:
    # Store the burned in model as pickle to reuse it
    with open(f"../snapshots/macro/abm_{macro_model._seed}.pkl", "wb") as f:
        pickle.dump(macro_model, f)

In [12]:
len(macro_models[0].learning_firms)

100